# @toolデコレータを使ってツールを定義する

## 1、ツールの説明をカスタマイズする：description

例1：

関数を@toolデコレータで修飾すると、モデルが認識できるツールになります。

以下のプログラムはエラーになります。理由は：
descriptionパラメータを指定しない場合、関数には必ずdocstringが必要だからです。

In [3]:
from anthropic import BaseModel
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as print


@tool
def get_weather(city: str):
    return f"{city}は晴れです"


print(convert_to_openai_tool(get_weather))

ValueError: Function must have a docstring if description not provided.

修正後：

In [4]:


from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool
def get_weather(city: str):
    """都市の天気を取得する"""
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を取得する',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

例2：description パラメータを使用

In [5]:


from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(description="具体的な都市の天気状況を取得する")
def get_weather(city: str):
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '具体的な都市の天気状況を取得する',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

例3：

description と docstring の両方を宣言した場合、description の優先度がより高くなります

In [6]:


from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(description="具体的な都市の天気状況を取得する")
def get_weather(city: str):
    """都市の天気を取得する"""
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '具体的な都市の天気状況を取得する',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

例4：

`@tool`に`description`パラメータを渡さない場合、デフォルトでは`tool`は`docstring`全体を`description`とみなします


`@tool`デコレータを使わない場合、不正な形式の docstring は普通のテキストとして扱われ`description`になりますが、`@tool`を使用している場合に docstring の形式が不正だと例外が発生します

In [7]:


from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str):
    """
    都市の天気を取得する

    Args:
        city : 都市
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を取得する',
        'parameters': {
            'properties': {'city': {'description': '都市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

例5：

In [8]:


from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(parse_docstring=True, description="具体的な都市の天気を取得する")
def get_weather(city: str):
    """
    都市の天気を取得する

    Args:
        city : 都市
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '具体的な都市の天気を取得する',
        'parameters': {
            'properties': {'city': {'description': '都市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2、ツール名の変更：name_or_callable

開発では、関数名をツール名として使うのが一般的で、ツール名をカスタマイズすることは推奨されません。

例：

In [9]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool(parse_docstring=True, name_or_callable="getWeather")
def get_weather(city: str):
    """
    都市の天気を取得する

    Args:
        city : 都市
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '都市の天気を取得する',
        'parameters': {
            'properties': {'city': {'description': '都市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 3、args_schema をカスタマイズする

### 3.1 Pydantic モデルを使って定義

例1：

In [10]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel


class WeatherInput(BaseModel):
    city: str


@tool(args_schema=WeatherInput)
def get_weather(city: str):
    """
    都市の天気を取得する
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を取得する',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

例2：

In [11]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        description="具体的な都市",
        default="北京",
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str):
    """
    都市の天気を取得する
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を取得する',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的な都市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

例3：

In [12]:
from typing import Literal


class WeatherInput(BaseModel):
    city: str = Field(
        description="具体的な都市",
        default="北京",
    )
    unit: Literal["celsius", "fahrenheit"]
    include_forecast: bool = Field(
        default=False,
        description="今後5日間の天気予報を含めるかどうか"
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = True):
    """
    都市の天気を取得する
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を取得する',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的な都市', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '今後5日間の天気予報を含めるかどうか',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

### 3.2 JSON Schema を使って定義

例：

In [13]:

json_schema = {
    'properties': {
        'city': {'default': '北京', 'description': '具体的な都市111', 'type': 'string'},
        'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
        'include_forecast': {
            'default': False,
            'description': '今後5日間の天気予報を含めるかどうか111',
            'type': 'boolean'
        }
    },
    'required': ['unit'],
    'type': 'object'
}


@tool(args_schema=json_schema)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = True):
    """
    都市の天気を取得する
    """
    return f"{city}は晴れです"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を取得する',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的な都市111', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '今後5日間の天気予報を含めるかどうか111',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}